In [33]:
# Install necessary packages
!pip install requests==2.31.0
!pip install google-cloud-aiplatform[adk] --upgrade

  Using cached requests-2.31.0-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.31.0-py3-none-any.whl (62 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.31.0 which is incompatible.
datasets 4.0.0 requires requests>=2.32.2, but you have requests 2.31.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.
libpysal 4.14.1 requires requests>=2.32.0, but you have requests 2.31.0 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.2

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
    state = resolution.resolve(requirements, max_

In [ ]:
import os
import requests
import json
from google.adk.agents import Agent
from google.adk.tools import google_search
from vertexai import agent_engines
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
import logging
from typing import Optional
from google.cloud import aiplatform
from google import genai
import vertexai
from vertexai.preview.generative_models import GenerativeModel
from google.adk.tools.agent_tool import AgentTool
from google.adk.agents import Agent

# --- API Keys ---
GOOGLE_MAPS_API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
NWS_USER_AGENT = "myweatherapp.com, contact@myweatherapp.com"
MODEL_ARMOR_API_KEY="YOUR_MODEL_ARMOR_API_KEY"

# Initialize AI Platform
PROJECT_ID = "YOUR_PROJECT_ID"
LOCATION = "us-central1"

# Set environment variables for ADK to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# Model Armor location
MODEL_ARMOR_LOCATION = "us"
MODEL_ARMOR_TEMPLATE_ID = "Challenge_2_Hsullivan"
model_armor_config = {
    "prompt_template": f"projects/{PROJECT_ID}/locations/{LOCATION}/templates/{MODEL_ARMOR_TEMPLATE_ID}"
}

# Initialize the Vertex AI client
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Initialize the genai client for ADK
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

In [56]:
# Initialize logger
logger = logging.getLogger(__name__)

def get_weather_forecast(location_name: str) -> str:
    """
    Fetches the weather forecast for a given location.

    Args:
        location_name: The name of the location (e.g., "Seattle", "London").

    Returns:
        A string containing the weather forecast or an error message.
    """
    # First, get coordinates for the location
    coords = get_coordinates_google_maps(location_name)
    if not coords:
        return f"Sorry, I couldn't find a location called '{location_name}'."

    latitude, longitude = coords

    # Then, get the NWS forecast using the coordinates
    return fetch_nws_forecast(latitude, longitude)

def get_coordinates_google_maps(location_name: str) -> tuple[float, float] | None:
    """
    Helper function to convert a location name into latitude and longitude.
    This function is NOT a tool itself but is used by the get_weather_forecast tool.
    """
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": location_name, "key": GOOGLE_MAPS_API_KEY}
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        if data["status"] == "OK" and data["results"]:
            location = data["results"][0]["geometry"]["location"]
            return location["lat"], location["lng"]
    except (requests.exceptions.RequestException, json.JSONDecodeError, KeyError):
        return None
    return None

def fetch_nws_forecast(latitude: float, longitude: float) -> str:
    """
    Helper function to fetch the forecast from the NWS.
    This function is NOT a tool itself but is used by the get_weather_forecast tool.
    """
    headers = {"User-Agent": NWS_USER_AGENT, "Accept": "application/geo+json"}
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    try:
        response = requests.get(points_url, headers=headers)
        response.raise_for_status()
        point_data = response.json()
        forecast_url = point_data.get("properties", {}).get("forecast")

        if not forecast_url:
            return "Could not find forecast URL."

        response = requests.get(forecast_url, headers={"User-Agent": NWS_USER_AGENT, "Accept": "application/json"})
        response.raise_for_status()
        forecast_data = response.json()
        periods = forecast_data.get("properties", {}).get("periods", [])

        if not periods:
            return "No forecast periods available."

        summary = [f"**{p.get('name')}**: {p.get('shortForecast')}. Temp: {p.get('temperature')}°{p.get('temperatureUnit')}." for p in periods[:3]]
        return "\n".join(summary)
    except (requests.exceptions.RequestException, json.JSONDecodeError, KeyError) as e:
        return f"Error fetching NWS data: {e}"

def web_search(query: str) -> str:
    """
    Searches the web using Google Search via Gemini grounding.

    Args:
        query: The search query string.

    Returns:
        A string containing search results or an error message.
    """
    from google.genai import types

    # Use the genai client with Google Search grounding
    # The correct way is to use Tool with google_search config
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=query,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
        )
    )

    # Extract the text response
    if response.candidates and response.candidates[0].content.parts:
        return response.candidates[0].content.parts[0].text
    return "No search results found."

def check_user_input(user_text: str) -> str:
    """
    Performs content moderation using Model Armor.
    Returns "BAD" if content is inappropriate, otherwise "GOOD".
    """
    import google.auth
    from google.auth.transport.requests import Request

    # Get credentials
    credentials, project = google.auth.default()
    credentials.refresh(Request())

    # Call Model Armor API
    url = f"https://modelarmor.{MODEL_ARMOR_LOCATION}.rep.googleapis.com/v1/projects/{PROJECT_ID}/locations/{MODEL_ARMOR_LOCATION}/templates/{MODEL_ARMOR_TEMPLATE_ID}:sanitizeUserPrompt"

    headers = {
        "Authorization": f"Bearer {credentials.token}",
        "Content-Type": "application/json"
    }

    data = {
        "userPromptData": {
            "text": user_text
        }
    }

    response = requests.post(url, headers=headers, json=data)
    response.raise_for_status()

    result = response.json()

    # Check if any filters matched
    filter_state = result.get("sanitizationResult", {}).get("filterMatchState", "")
    if filter_state == "MATCH_FOUND":
        return "BAD"

    return "GOOD"

def log_model_response (callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
  """
  Writes the first text part of the model's response to the log and passes the response through unchained.
  """
  if llm_response.content and llm_response.content.parts:
    txt = llm_response.content.parts[0].text
    if txt:
      logger.info("[%s] MODEL >> %s", callback_context.agent_name, txt.strip())
  return None

def chained_before_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
  # 1. Moderation check
  moderation_result = moderate_user_prompt(callback_context, llm_request)
  if moderation_result is not None:
    return moderation_result

  # 2. Log user input
  log_user_prompt(callback_context, llm_request)
  return None

def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            logger.info("[%s] USER .. %s", callback_context.agent_name,
            last.parts[0].text.strip())
    return None

def moderate_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    try:
        if not llm_request.contents:
            return None
        last = llm_request.contents[-1]
        user_text = last.parts[0].text.strip()
        result = check_user_input(user_text)

        if result == "BAD":
            return LlmResponse(content={"role": "model", "parts": [{"text": "Message violates our content guidelines."}]})
        # If GOOD, return None to proceed
    except Exception as e:
        logger.exception("Error moderating user prompt")
    return None

In [53]:
# Weather sub-agent - has custom function tool
weather_agent = Agent(
    model="gemini-2.0-flash",
    name='weather_agent',
    description="Handles weather-related queries. Use this agent when the user asks about weather forecasts, temperature, or climate for any location.",
    instruction="You are a weather assistant. Use the get_weather_forecast tool to fetch weather information for locations the user asks about.",
    tools=[get_weather_forecast],
)

# Search sub-agent - uses custom web_search function (Python callable, compatible with AFC)
search_agent = Agent(
    model="gemini-2.0-flash",
    name='search_agent',
    description="Handles general knowledge queries, news, and web searches. Use this agent for any non-weather questions that require searching the internet.",
    instruction="You are a search assistant. Use the web_search tool to find information for the user's query and provide a helpful summary.",
    tools=[web_search],
)

# Root agent that orchestrates sub-agents (no tools, just routing)
root_agent = Agent(
    model="gemini-2.0-flash",
    name='assistant',
    instruction="""You are a helpful assistant that routes user requests to the appropriate specialist agent.

For weather-related questions (forecasts, temperature, climate, etc.), delegate to weather_agent.
For all other questions (news, general knowledge, searches), delegate to search_agent.

Always delegate to the appropriate agent - do not try to answer directly.""",
    description="A helpful assistant that can answer weather questions and search the web for information.",
    sub_agents=[weather_agent, search_agent],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

# Create an AdkApp instance for local testing
app = agent_engines.AdkApp(agent=root_agent)

In [57]:
# --- Interactive Chat Loop ---
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import asyncio

# Create runner for weather agent only (search is handled directly)
session_service = InMemorySessionService()
weather_runner = Runner(agent=weather_agent, app_name="weather_app", session_service=session_service)

def is_weather_query(text: str) -> bool:
    """Simple heuristic to detect weather-related queries."""
    weather_keywords = ['weather', 'forecast', 'temperature', 'rain', 'sunny', 'cloudy', 'snow', 'hot', 'cold', 'humid', 'climate']
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in weather_keywords)

async def run_chat_async():
    print("Assistant is ready. You can ask about weather or search the web. Type 'bye' to exit.")

    # Create session for weather agent
    weather_session = await session_service.create_session(app_name="weather_app", user_id="user1")

    while True:
        user_input = input("You: ")
        if user_input.lower() == "bye":
            print("It was nice chatting with you!")
            break

        # Check user input with Model Armor before processing
        try:
            moderation_result = check_user_input(user_input)
            print(f"Moderation Result: {moderation_result}")
            if moderation_result == "BAD":
                print("Assistant: Message violates our content guidelines.")
                continue
        except Exception as e:
            logger.warning(f"Moderation check failed: {e}. Proceeding with request.")

        # Route to appropriate agent based on query type
        try:
            if is_weather_query(user_input):
                print("[Routing to: weather_agent]")
                content = types.Content(role="user", parts=[types.Part(text=user_input)])
                response_parts = []
                async for event in weather_runner.run_async(user_id="user1", session_id=weather_session.id, new_message=content):
                    if event.content and event.content.parts:
                        for part in event.content.parts:
                            if hasattr(part, 'text') and part.text:
                                response_parts.append(part.text)
                final_answer = " ".join(response_parts) if response_parts else "No response from agent."
            else:
                print("[Routing to: search_agent]")
                # Call web_search directly (uses Google Search grounding)
                final_answer = web_search(user_input)

            print(f"Assistant: {final_answer}")
        except Exception as e:
            print(f"Assistant: Error processing request: {e}")

# Start the chat
await run_chat_async()

Assistant is ready. You can ask about weather or search the web. Type 'bye' to exit.
You: tell me about the weather in dallas
Moderation Result: GOOD
[Routing to: weather_agent]
Assistant: OK. This afternoon in Dallas it will be partly sunny with a temperature of 80°F. Tonight, it will be partly cloudy with a temperature of 63°F. Tuesday will be partly sunny with a temperature of 84°F.

You: tell me about mocking birds
Moderation Result: GOOD
[Routing to: search_agent]
Assistant: The Northern Mockingbird ( *Mimus polyglottos*) is a medium-sized songbird native to North America. It's also found in parts of the Caribbean and the Hawaiian Islands. The mockingbird is known for its impressive vocal abilities, territorial behavior, and adaptability to human environments.

Here's some more information about mockingbirds:

**Description:**

*   Mockingbirds have gray-brown upper feathers and a paler belly.
*   Their wings have white patches, which are visible during flight.
*   Adults are 8.3-

In [58]:
# Test Model Armor directly
print("--- Testing Model Armor ---")

test_prompts = [
    "What is the weather in Seattle?",  # Should pass (GOOD)
    "Tell me a joke",                    # Should pass (GOOD) - off-topic but not harmful
    "How do I hack into a computer?",    # May be flagged (BAD) depending on template config
]

for prompt in test_prompts:
    try:
        result = check_user_input(prompt)
        print(f"Prompt: '{prompt}' -> Result: {result}")
    except Exception as e:
        print(f"Prompt: '{prompt}' -> Error: {e}")

print("--- End Model Armor Test ---")

--- Testing Model Armor ---
Prompt: 'What is the weather in Seattle?' -> Result: GOOD
Prompt: 'Tell me a joke' -> Result: GOOD
Prompt: 'How do I hack into a computer?' -> Result: BAD
--- End Model Armor Test ---


In [59]:
# Test code to check weather for multiple cities
cities_to_test = ["New York", "Miami", "Los Angeles"]

print("--- Testing Weather Forecast for Multiple Cities ---")
for city in cities_to_test:
    print(f"\nWeather for {city}:")
    forecast = get_weather_forecast(city)
    print(forecast)
print("----------------------------------------------------")

--- Testing Weather Forecast for Multiple Cities ---

Weather for New York:
**This Afternoon**: Mostly Sunny. Temp: 32°F.
**Tonight**: Partly Cloudy then Slight Chance Light Snow. Temp: 31°F.
**Tuesday**: Chance Light Snow then Rain And Snow. Temp: 39°F.

Weather for Miami:
**This Afternoon**: Mostly Sunny. Temp: 78°F.
**Tonight**: Partly Cloudy. Temp: 73°F.
**Tuesday**: Slight Chance Rain Showers. Temp: 78°F.

Weather for Los Angeles:
**Today**: Sunny. Temp: 76°F.
**Tonight**: Partly Cloudy then Patchy Fog. Temp: 50°F.
**Tuesday**: Patchy Fog then Mostly Sunny. Temp: 78°F.
----------------------------------------------------


In [60]:
# Test the multi-agent system
print("--- Testing Multi-Agent System ---\n")

# Test cases for the root agent delegation
test_queries = [
    # Weather queries - should delegate to weather_agent
    {"query": "What's the weather in Chicago?", "expected_agent": "weather_agent"},
    {"query": "Will it rain in Boston tomorrow?", "expected_agent": "weather_agent"},

    # Search queries - should delegate to search_agent
    {"query": "Search for the latest news about AI", "expected_agent": "search_agent"},
    {"query": "Find information about Python programming", "expected_agent": "search_agent"},

    # Ambiguous query - could go either way
    {"query": "What's happening in Seattle today?", "expected_agent": "either"},
]

print("Testing weather_agent directly:")
print("-" * 40)
weather_result = get_weather_forecast("Denver")
print(f"Weather in Denver:\n{weather_result}\n")

print("Testing search_agent directly:")
print("-" * 40)
try:
    # Test google_search tool
    search_result = google_search("current time in Tokyo")
    print(f"Search result: {search_result}\n")
except Exception as e:
    print(f"Search error (may need API setup): {e}\n")

print("Testing Model Armor with agent queries:")
print("-" * 40)
for test in test_queries:
    query = test["query"]
    try:
        moderation = check_user_input(query)
        print(f"Query: '{query}'")
        print(f"  Moderation: {moderation}")
        print(f"  Expected agent: {test['expected_agent']}")
        print()
    except Exception as e:
        print(f"Query: '{query}' -> Error: {e}\n")

print("--- End Multi-Agent Tests ---")

--- Testing Multi-Agent System ---

Testing weather_agent directly:
----------------------------------------
Weather in Denver:
**This Afternoon**: Partly Sunny. Temp: 72°F.
**Tonight**: Partly Cloudy then Chance Light Rain. Temp: 38°F.
**Tuesday**: Light Rain Likely. Temp: 48°F.

Testing search_agent directly:
----------------------------------------
Search error (may need API setup): 'GoogleSearchTool' object is not callable

Testing Model Armor with agent queries:
----------------------------------------
Query: 'What's the weather in Chicago?'
  Moderation: GOOD
  Expected agent: weather_agent

Query: 'Will it rain in Boston tomorrow?'
  Moderation: GOOD
  Expected agent: weather_agent

Query: 'Search for the latest news about AI'
  Moderation: GOOD
  Expected agent: search_agent

Query: 'Find information about Python programming'
  Moderation: GOOD
  Expected agent: search_agent

Query: 'What's happening in Seattle today?'
  Moderation: GOOD
  Expected agent: either

--- End Multi-